# Averaging the past while ignoring the future (code)

In [1]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

## Equal-weighted average of the past

In [2]:
thepast = torch.tensor( [4,1,-2,-3] )
N = len(thepast)

weights = torch.ones(N) / N

print(f'The past: {thepast}')
print(f'Weights (importance) of the past: {weights}')
print(f'Sum over all weights: {sum(weights)}')

The past: tensor([ 4,  1, -2, -3])
Weights (importance) of the past: tensor([0.2500, 0.2500, 0.2500, 0.2500])
Sum over all weights: 1.0


In [3]:
thepresent = sum(thepast * weights)
print(f'The present (weighted sum of the past): {thepresent}')

The present (weighted sum of the past): 0.0


## Weighted average of the past

In [4]:
weights = torch.tensor( [2,1,1,1] )
print(f'Sum of weights: {sum(weights)}. Uh oh...')

Sum of weights: 5. Uh oh...


In [5]:
linear_weights = weights / sum(weights)
softmax_weights = torch.exp(weights) / torch.exp(weights).sum()

print(f'Scaled weights: {linear_weights}')
print(f'\tTheir sum: {sum(linear_weights)}')

print(f'\nSoftmax weights: {softmax_weights}')
print(f'\tTheir sum: {sum(softmax_weights)}')

Scaled weights: tensor([0.4000, 0.2000, 0.2000, 0.2000])
	Their sum: 1.0

Softmax weights: tensor([0.4754, 0.1749, 0.1749, 0.1749])
	Their sum: 1.0


In [6]:
thepresent_linear = sum(thepast * linear_weights)
thepresent_softmax = sum(thepast * softmax_weights)

print(f'The present (linear sum of the past):  {thepresent_linear}')
print(f'The present (softmax sum of the past): {thepresent_softmax}')

The present (linear sum of the past):  0.8000000715255737
The present (softmax sum of the past): 1.2019567489624023


## Ignoring the future

In [7]:
thedata = torch.tensor( [4,1,-2,-3,8,3,-1] )
present_moment = 4
N = len(thedata)

print(f'Past data: {thedata[:present_moment]}')
print(f'The present: {thedata[present_moment]}')
print(f'The future: {thedata[present_moment+1:]}')

Past data: tensor([ 4,  1, -2, -3])
The present: 8
The future: tensor([ 3, -1])


In [8]:
past_weights = torch.ones(N)
past_weights[present_moment+1:] = 0
past_weights

tensor([1., 1., 1., 1., 1., 0., 0.])

In [9]:
past_weights_linear = past_weights / torch.sum(past_weights)

print(f'Scaled weights: {past_weights_linear}')
print(f'\tTheir sum: {sum(past_weights_linear)}')

Scaled weights: tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000])
	Their sum: 1.0


In [10]:
# softmax the weights with zeros
past_weights_softmax = torch.exp(past_weights) / torch.sum(torch.exp(past_weights))

print(f'Softmax weights: {past_weights_softmax}')
print(f'\tTheir sum: {sum(past_weights_softmax)}')

Softmax weights: tensor([0.1743, 0.1743, 0.1743, 0.1743, 0.1743, 0.0641, 0.0641])
	Their sum: 1.0


In [11]:
# e.g.:
torch.exp(torch.tensor([-10]))

tensor([4.5400e-05])

In [12]:
# recreate the weights for the past, but setting future values to -infinity
past_weights = torch.ones(N)
past_weights[present_moment+1:] = -torch.inf

# softmaxify
past_weights_softmax = torch.exp(past_weights) / torch.sum(torch.exp(past_weights))

# print the results
print(f'Unscaled weights: {past_weights}')
print(f'Scaled weights: {past_weights_softmax}')
print(f'\tTheir sum: {sum(past_weights_softmax)}')

Unscaled weights: tensor([1., 1., 1., 1., 1., -inf, -inf])
Scaled weights: tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000])
	Their sum: 1.0


## Steps toward the future, looking back into the past

In [13]:
# rows are calculated steps, columns are time points
tril = torch.tril(torch.ones(9,9))
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1.]])

In [14]:
tril[tril==0] = -torch.inf
tril

tensor([[1., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., 1., -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., 1., 1., -inf, -inf, -inf],
        [1., 1., 1., 1., 1., 1., 1., -inf, -inf],
        [1., 1., 1., 1., 1., 1., 1., 1., -inf],
        [1., 1., 1., 1., 1., 1., 1., 1., 1.]])

In [15]:
# softmaxify
tril_softmax = F.softmax(tril, dim=-1)
tril_softmax

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.0000],
        [0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111]])

In [16]:
for timepoint in range(tril.shape[0]):
    print(f'\nWeights for calculation at time point {timepoint}:')
    print(f'\t{tril_softmax[timepoint]}')


Weights for calculation at time point 0:
	tensor([1., 0., 0., 0., 0., 0., 0., 0., 0.])

Weights for calculation at time point 1:
	tensor([0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

Weights for calculation at time point 2:
	tensor([0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

Weights for calculation at time point 3:
	tensor([0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

Weights for calculation at time point 4:
	tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000, 0.0000])

Weights for calculation at time point 5:
	tensor([0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000, 0.0000])

Weights for calculation at time point 6:
	tensor([0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000, 0.0000])

Weights for calculation at time point 7:
	tensor([0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.0000])

Weights for calculation at time point 8:
	t

## Final demo with random activations

In [17]:
activations = torch.randn(N,N)
tril = torch.tril(torch.ones(N,N))

print('-- ORIGINAL ACTIVATIONS:')
print(activations)

print('\n-- PAST WEIGHTING FACTOR:')
print(tril)

scaled_activations = activations * tril
scaled_activations[scaled_activations==0] = -torch.inf
print('\n-- SCALED PAST ACTIVATIONS:')
print(scaled_activations)

softmax_past = F.softmax(scaled_activations, dim=-1)
print('\n-- SOFTMAX PAST ACTIVATIONS:')
print(softmax_past)

-- ORIGINAL ACTIVATIONS:
tensor([[ 0.2900, -0.9867, -0.0064,  1.1481, -1.9467,  1.1210,  0.5094],
        [ 0.4760, -0.5283, -1.0650, -0.6716, -0.2813, -0.3057, -0.3932],
        [-0.7794,  0.4846,  0.6434, -0.2599,  0.5483,  0.3808, -0.7312],
        [ 0.9009, -0.2191, -0.8758,  1.6025, -0.9152,  0.1565,  0.4436],
        [ 0.4462, -0.1063,  0.9903,  0.1481,  0.5913, -0.1759, -0.1086],
        [ 1.4774,  1.6591, -2.0570, -2.1638, -2.8551,  0.6148,  1.9285],
        [-0.9178,  0.7222,  0.1625,  1.0607,  0.7269, -1.3588, -1.2585]])

-- PAST WEIGHTING FACTOR:
tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1.]])

-- SCALED PAST ACTIVATIONS:
tensor([[ 0.2900,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.4760, -0.5283,    -inf,    -inf,    -inf,    -inf,    -

In [18]:
# confirm:
torch.sum(softmax_past, dim=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

## FYI, timing some alternatives

In [19]:
import time
nIters = int(2e5)

# option 1: find zeros and -torch.inf
start_time = time.time()
for _ in range(nIters):
  tril = torch.tril(torch.ones(10,10))
  tril[tril==0] = -torch.inf
print(f'Option 1: {time.time()-start_time:.3f} sec')

# option 2: find zeros and float('-inf')
start_time = time.time()
for _ in range(nIters):
  tril = torch.tril(torch.ones(10,10))
  tril[tril==0] = float('-inf')
print(f'Option 2: {time.time()-start_time:.3f} sec')

# option 3: masked_fill with float('-inf')
start_time = time.time()
for _ in range(nIters):
  tril = torch.tril(torch.ones(10,10))
  tril = tril.masked_fill(tril==0, float('-inf'))
print(f'Option 3: {time.time()-start_time:.3f} sec')

Option 1: 8.006 sec
Option 2: 8.081 sec
Option 3: 7.984 sec
